In [6]:
"""
Project: "Deep-learning-based decomposition of overlapping-sparse images:
          application at the vertex of neutrino interactions"
Paper: https://arxiv.org/abs/2310.19695.
Author: Dr. Saul Alonso-Monsalve
Contact: salonso@ethz.ch/saul.alonso.monsalve@cern.ch
Description: Training script for the proton GAN considering the first transformer configuration.
"""

import sys, os
sys.path.append(os.path.abspath("..")) 
import json
import torch
import pytorch_lightning as pl
# check lightning version
print(pl.__version__)

from torch.utils.data import DataLoader
from datasets import GANDataset
from models import Generator, Critic, WGAN_GP_Loss, LightningModelGAN
from utils import args_gan
from pytorch_lightning.loggers import CSVLogger
from pytorch_lightning.callbacks import ModelCheckpoint



2.5.5


In [2]:
# Manually specify the GPUs to use
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.multiprocessing.set_sharing_strategy('file_system')
# Arguments
parser = args_gan()
args, unknown = parser.parse_known_args()

args.particle = "proton_contained"
args.metadata_path = "/scratch/libota/sfgd_va_nn_data/NN_Data/metadata.pkl"
args.dataset_path = "/scratch/libota/sfgd_va_nn_data/NN_Data/{}/{}/{}.npz"
args.gan_ind_path = "/scratch/libota/sfgd_va_nn_data/NN_Data/gan_ind.pkl"
args.save_dir = "/scratch2/libota/SFGD_Vertex_Activity/Results/gan/"
args.checkpoint_path = "/scratch2/libota/SFGD_Vertex_Activity/Results/gan/checkpoints"
args.checkpoint_name = "proton_contained"

args.epochs = 50
args.log_every_n_steps = 2000
args.batch_size = 2048
args.hidden = 64
args.warmup_steps = 10
args.num_workers = 64

    

In [3]:

# Training set and loader
train_set = GANDataset(args, split="train")
train_loader = DataLoader(train_set, collate_fn=train_set.collate_fn, batch_size=args.batch_size,
                          num_workers=args.num_workers, shuffle=True)



In [4]:
# Geneator and critic models
generator = Generator(input_size=args.input_size, label_size=args.label_size, noise_size=args.noise_size,
                      hidden=args.hidden, n_layers=args.layers, attn_heads=args.attn_heads, dropout=args.dropout)
critic = Critic(input_size=args.input_size, label_size=args.label_size, noise_size=args.noise_size,
                    hidden=args.hidden, n_layers=args.layers, attn_heads=args.attn_heads, dropout=args.dropout)
    

In [5]:
generator._init_weights()
critic._init_weights()
gen_total_params = sum(p.numel() for p in generator.parameters() if p.requires_grad)
cri_total_params = sum(p.numel() for p in critic.parameters() if p.requires_grad)
print(generator)
print(critic)
print("Total trainable params: {} (generator), {} (discriminator).".format(gen_total_params, cri_total_params))



Generator(
  (bert): BERT(
    (embedding): Embedding(
      (input): Linear(in_features=1, out_features=64, bias=True)
      (label): LabelEmbedding(
        (embedding): Linear(in_features=6, out_features=64, bias=True)
      )
      (position): PositionalEmbedding(
        (embedding): Embedding(126, 64)
      )
      (noise): NoiseEmbedding(in_features=512, out_features=64, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer_blocks): ModuleList(
      (0-1): 2 x TransformerBlock(
        (attention): MultiHeadedAttention(
          (linear_layers): ModuleList(
            (0-2): 3 x Linear(in_features=64, out_features=64, bias=True)
          )
          (output_linear): Linear(in_features=64, out_features=64, bias=True)
          (attention): Attention()
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (feed_forward): PositionwiseFeedForward(
          (w_1): Linear(in_features=64, out_features=256, bias=True)
          (w_2): Linea

In [ ]:
# Loss function (for critic)
adv_loss = WGAN_GP_Loss(args.lambda_gp)

# Create lightning model
lightning_model = LightningModelGAN(generator=generator,
                                    critic=critic,
                                    noise_size=args.noise_size,
                                    crit_repeats=args.crit_repeats,
                                    adversarial_loss=adv_loss, lr=args.lr, wd=args.weight_decay)

# Define logger and checkpoint
logger = CSVLogger(save_dir="logs/", name=config["log_path"])
checkpoint_callback = ModelCheckpoint(dirpath=config["save_path"], every_n_train_steps=5000)

# Create trainer module
trainer = pl.Trainer(
    max_epochs=args.epochs,
    callbacks=[checkpoint_callback],
    accelerator="gpu",
    precision="bf16",
    devices=[0],
    logger=logger,
    log_every_n_steps=100,
    deterministic=True,
)

# Run the training
trainer.fit(
    model=lightning_model,
    train_dataloaders=train_loader,
)